In [2]:
# packages and working directory  
import scipy 
import sklearn
import econml 
import arch
import os 
import pandas as pd 
import numpy as np 
import seaborn as sns 
import matplotlib as plt
import statsmodels.api as sm 
from statsmodels.discrete.discrete_model import Probit
from statsmodels.iolib.summary2 import summary_col
import statsmodels.formula.api as smf 
from scipy.optimize import minimize
from scipy.special import logsumexp
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt 
from scipy import stats
from scipy.stats import ttest_ind
from scipy.optimize import approx_fprime
# Consolidate changing directory and CPI dictionary since these don't change throughout the script
new_directory = r'C:\Users\hisham\Spain\2021 datasets'
os.chdir(new_directory)


In [3]:
df = pd.read_csv('filtered.csv')
df.head()

,idhh,idperson,idmother,idfather,idpartner,idorighh,idorigperson,dag,dgn,dec,...,il_bsarg_63,il_bsarg_64,il_bsarg_70,scenario,original_scenario,choice_made,lhw_h0,lhw_h1,lhw_h2,lhw_h3
0,870100,87010001,0,0,87010002,870100,87010001,65,1,0,...,0.00,0.00,0.00,h0,h2,0,0,7,42,54
1,870600,87060001,0,0,0,870600,87060001,32,1,0,...,469.93,469.93,469.93,h0,h2,0,0,11,44,50
2,870700,87070001,0,0,0,870700,87070001,61,0,0,...,1747.61,1747.61,1747.61,h0,h0,1,0,7,27,52
3,870900,87090001,0,0,0,870900,87090001,33,0,0,...,100.00,100.00,100.00,h0,h2,0,0,10,37,51
4,871700,87170002,0,0,87170001,871700,87170002,61,0,0,...,312.50,312.50,312.50,h0,h2,0,0,7,40,51


In [4]:
headers = df.columns.tolist()

headers

['idhh',
 'idperson',
 'idmother',
 'idfather',
 'idpartner',
 'idorighh',
 'idorigperson',
 'dag',
 'dgn',
 'dec',
 'dwt',
 'dms',
 'deh',
 'drgn2',
 'ddi',
 'dlg_s',
 'ddilv',
 'dcz',
 'drgur',
 'drgmd',
 'drgru',
 'ddt',
 'dsu00',
 'dsu01',
 'dsu02',
 'dncsy',
 'dmb',
 'dct',
 'dehde',
 'dey',
 'drgn1',
 'dsr',
 'les',
 'loc',
 'loopcount_pens',
 'liwft_s',
 'lhw',
 'lindi',
 'lhwsr_s',
 'lhwsesr_s',
 'lunmy_s',
 'lunmy',
 'liwmy_s',
 'liwwh',
 'liwmy_a',
 'lnu',
 'lhwpv_a',
 'liwmy02_a',
 'lfs',
 'lcs',
 'liwmy',
 'liwftmy',
 'liwptmy',
 'lpemy',
 'lse',
 'liwmy_f',
 'lhw_f',
 'liwwh_f',
 'lunmy_f',
 'yem',
 'yse',
 'yemmc_s',
 'yiy',
 'yot',
 'ypr',
 'ypt',
 'ypp',
 'yemxm_s',
 'yemmy',
 'ysemy',
 'yemmw_s',
 'ysemw_s',
 'ysemc_s',
 'yempv_s',
 'yivwg',
 'yempv_a',
 'ysv',
 'yptmp',
 'yds',
 'ydses_o',
 'poa00',
 'pdi00',
 'pdicm',
 'pdinc',
 'psuwd00',
 'poacm',
 'poanc',
 'poaot',
 'poa',
 'pdiot',
 'pdi',
 'psuwdcm',
 'psuot',
 'psu',
 'poanc00_s',
 'poancna_s',
 'poancrg_s',
 

In [5]:
df['dec'].describe()

count    18320.000000
mean         0.235371
std          1.111136
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          6.000000
Name: dec, dtype: float64

In [6]:
excluded_vars = [ 'dec',
 'dwt',
 'dms',
 'deh',
 'drgn2',
 'ddi',
 'dlg_s',
 'ddilv',
 'dcz',
 'drgur',
 'drgmd',
 'drgru',
 'ddt',
 'dncsy',
 'dmb',
 'dct',
 'dehde',
 'dey',
 'drgn1',
 'dsr',
 'loc',
 'lnu',
 'lfs',
 'lcs',
 'lse',
 'yse',
 'yiy',
 'yot',
 'ypr',
 'ypt',
 'ypp',
 'yivwg',
 'ysv',
 'yptmp',
 'poa00',
 'pdi00',
 'pdicm',
 'pdinc',
 'psuwd00',
 'poacm',
 'poanc',
 'poaot',
 'poa',
 'pdiot',
 'pdi',
 'psuwdcm',
 'psuot',
 'psu',
 'pdimy',
 'poamy',
 'psumy',
 'bsarg_s',
 'bsa00_s',
 'bhl',
 'bma',
 'bunct',
 'bunctmy',
 'bunctpc',
 'bunnc',
 'bunncmy',
 'bed',
 'bsa',
 'bunot',
 'bho',
 'bunct02_s',
 'bsa_s',
 'bun',
 'bfa',
 'bhl00',
 'bhlot',
 'bunmy',
 'bunotmy',
 'bch00',
 'bchdi',
 'bchot',
 'bch',
 'xmp',
 'xpp',
 'xhcmomi',
 'xhcmomc',
 'xhcmo',
 'xhcrt',
 'xcc',
 'xhc',
 'xed00',
 'xhl00',
 'xhcot',
 'tpr',
 'tad',
 'twl',
 'tscer',
 'tis',
 'tintrch',
 'kfb',
 'kfbcc',
 'kfbmy',
 'amrtn',
 'afc',
 'amrrm',
 'aco',
 'aca',
 'il_bsa00']

In [11]:
# Assuming these are your observed heterogeneity variables based on your dataset
observed_heterogeneity_vars = [
 'dag',
 'dgn',
]

# Initialize observed heterogeneity effects to zero for simplicity
observed_heterogeneity = {var: 0 for var in observed_heterogeneity_vars}


In [7]:
def utility(c, l, params, row):
    alpha1, alpha2, beta1, beta2, gamma = params[:5]
    # Compute the observed heterogeneity effect
    observed_effect = sum(row[var] * params[i+5] for i, var in enumerate(observed_heterogeneity_vars))
    # Placeholder for unobserved heterogeneity effect on alpha1
    zeta_effect = 0  # Modify as needed to incorporate unobserved heterogeneity
    
    return (alpha1 + zeta_effect) * c + alpha2 * c**2 + beta1 * l + beta2 * l**2 + gamma * c * l + observed_effect

In [8]:
def ind_likelihood(params, row, scenarios=['h0', 'h1', 'h2', 'h3']):
    utilities = [utility(row['ils_udb_yds'], 80 - row[f'lhw_{scenario}'], params, row) for scenario in scenarios]
    chosen_index = scenarios.index(row['original_scenario'])
    log_sum_exp_utilities = logsumexp(utilities)
    log_prob = utilities[chosen_index] - log_sum_exp_utilities
    return -log_prob  # Negative log-likelihood

def total_likelihood(params, df):
    return sum(df.apply(lambda row: ind_likelihood(params, row), axis=1))




In [9]:
from scipy.optimize import approx_fprime

def jacobian(params, df):
    epsilon = np.sqrt(np.finfo(float).eps)
    grad = approx_fprime(params, lambda p: total_likelihood(p, df), epsilon)
    return grad


In [13]:
from scipy.optimize import minimize

# Initial guesses for parameters, including those for observed heterogeneity
initial_params = [7.660e-01,  9.197e-01,  1.135e-01, -1.992e-03,  2.279e-05]  + [0.01] * len(observed_heterogeneity_vars)  # Adjust as necessary

result = minimize(lambda params: total_likelihood(params, df), initial_params, method='Newton-CG', jac=lambda params: jacobian(params, df))
print(result)



 message: Optimization terminated successfully.
 success: True
  status: 0
     fun: 19266.986439510663
       x: [ 7.660e-01  9.197e-01  1.135e-01 -1.901e-03  2.105e-05
            1.000e-02  1.000e-02]
     nit: 2
     jac: [ 0.000e+00  0.000e+00 -1.015e+05 -7.886e+06 -7.859e+07
            0.000e+00  0.000e+00]
    nfev: 6
    njev: 12
    nhev: 0


In [41]:
def calculate_utilities(row, params):
    # Assuming 'c' and 'l' can be directly extracted or calculated from the row
    # Here you need to specify how 'c' and 'l' should be calculated from the row
    c = row['ils_udb_yds']
    # This is an example for 'l', adjust it according to your needs
    l = 80 - row[f'lhw_{row["scenario"]}']
    return utility(c, l, params, row)


In [43]:
# Make sure df is defined and contains the necessary columns
# Ensure exploration_params is defined and correctly structured
# Example parameter values for exploration
exploration_params = [.766e+00,  9.197e-01,  1.135e-01, -1.993e-03,  1.893e-05] + [0.01] * len(observed_heterogeneity_vars)
df['computed_utility'] = df.apply(calculate_utilities, axis=1, args=(exploration_params,))

# Print the first few rows to verify the computed utilities
print(df[['idperson', 'scenario', 'computed_utility']].head())


   idperson scenario  computed_utility
0  87010001       h0      6.272085e+05
1  87060001       h0      6.502002e+05
2  87070001       h0      2.066611e+06
3  87090001       h0      6.183836e+05
4  87170002       h0      5.751697e+05


In [44]:
estimates = df[['idperson','ils_udb_yds','choice_made','computed_utility', 'scenario', 'lhw_h0','lhw_h1','lhw_h2','lhw_h3','lhw','yem','yivwg']]

estimates.to_csv('newesti.csv')

In [47]:
model_params = [.766e+00,  9.197e-01,  1.135e-01, -1.993e-03,  1.893e-05] + [0.01] * len(observed_heterogeneity_vars)


In [49]:
# This is a simplified example. You'll need to adjust this to accurately reflect your data structure and model.

# Assuming 'model_params' is a list or array of parameters for your model
df['likelihood'] = df.apply(
    lambda row: ind_likelihood(
        model_params,  # The parameters for your model
        row  # The entire row of the DataFrame
    ), 
    axis=1
)




In [53]:
import numpy as np

df['probability'] = np.exp(-df['likelihood'])



In [54]:
df[['idperson','probability','likelihood','ils_udb_yds','choice_made','computed_utility', 'scenario', 'lhw_h0','lhw_h1','lhw_h2','lhw_h3','lhw','yem','yivwg']].head(10)


,idperson,probability,likelihood,ils_udb_yds,choice_made,computed_utility,scenario,lhw_h0,lhw_h1,lhw_h2,lhw_h3,lhw,yem,yivwg
0,87010001,0.005722,5.163456,825.40,0,6.272085e+05,h0,0,7,42,54,42,0.0,13.85
1,87060001,0.005417,5.218274,840.40,0,6.502002e+05,h0,0,11,44,50,44,0.0,17.05
2,87070001,0.012925,4.348601,1498.60,1,2.066611e+06,h0,0,7,27,52,0,0.0,6.68
3,87090001,0.005887,5.135088,819.57,0,6.183836e+05,h0,0,10,37,51,37,0.0,25.68
4,87170002,0.005533,5.196968,790.40,0,5.751697e+05,h0,0,7,40,51,40,0.0,19.96
5,87350001,0.004453,5.414067,497.83,0,2.283126e+05,h0,0,7,40,55,40,0.0,14.99
6,87380001,0.004867,5.325373,525.00,0,2.538920e+05,h0,0,7,35,51,35,0.0,12.01
7,87490003,0.004290,5.451557,462.50,0,1.970812e+05,h0,0,9,40,55,40,0.0,6.82
8,87700002,0.005557,5.192657,548.27,0,2.768793e+05,h0,0,7,30,50,30,0.0,6.35
9,87840002,0.006804,4.990228,655.29,0,3.954234e+05,h0,0,20,27,60,20,0.0,13.29


In [57]:
# Assuming 'df' is your DataFrame
df_sorted = df.sort_values(by='idperson', ascending=True)

# If you want the sorting to be in place without creating a new DataFrame:
# df.sort_values(by='idperson', ascending=True, inplace=True)

# Display the first few rows to verify the sorting
df_sorted[['idperson','probability','likelihood','ils_udb_yds','choice_made','computed_utility', 'scenario', 'lhw_h0','lhw_h1','lhw_h2','lhw_h3','lhw','yem','yivwg']].head(10)




,idperson,probability,likelihood,ils_udb_yds,choice_made,computed_utility,scenario,lhw_h0,lhw_h1,lhw_h2,lhw_h3,lhw,yem,yivwg
0,87010001,0.005722,5.163456,825.40,0,6.272085e+05,h0,0,7,42,54,42,0.00,13.85
9160,87010001,0.535728,0.624128,2013.19,1,3.729029e+06,h2,0,7,42,54,42,2520.70,13.85
4580,87010001,0.019588,3.932827,825.40,0,6.272097e+05,h1,0,7,42,54,42,420.12,13.85
13740,87010001,0.366549,1.003622,2485.32,0,5.682724e+06,h3,0,7,42,54,42,3240.90,13.85
9161,87060001,0.480483,0.732963,2421.29,1,5.393733e+06,h2,0,11,44,50,44,3250.00,17.05
1,87060001,0.005417,5.218274,840.40,0,6.502002e+05,h0,0,11,44,50,44,0.00,17.05
4581,87060001,0.034210,3.375244,840.40,0,6.502020e+05,h1,0,11,44,50,44,812.50,17.05
13741,87060001,0.390829,0.939485,2686.71,0,6.640834e+06,h3,0,11,44,50,44,3693.18,17.05
2,87070001,0.012925,4.348601,1498.60,1,2.066611e+06,h0,0,7,27,52,0,0.00,6.68
13742,87070001,0.446935,0.805342,2547.16,0,5.968990e+06,h3,0,7,27,52,0,1504.36,6.68


In [60]:
import pandas as pd

# Assuming 'df' is your DataFrame with an 'idperson', 'scenario', 'probability', and 'computed_utility' columns

# Step 1: Determine the predicted_choice based on the highest probability
# Group by 'idperson' and find the row with the max probability for each group
predicted_choice = df_sorted.loc[df.groupby('idperson')['probability'].idxmax()]

# Create a binary column for the predicted choice
predicted_choice['predicted_choice'] = 1

# Step 2: Determine the optimal_choice based on the highest computed_utility
# Group by 'idperson' and find the row with the max computed_utility for each group
optimal_choice = df_sorted.loc[df.groupby('idperson')['computed_utility'].idxmax()]

# Create a binary column for the optimal choice
optimal_choice['optimal_choice'] = 1

# Merge the predicted_choice and optimal_choice back to the original DataFrame
df_sorted = df_sorted.merge(predicted_choice[['idperson', 'scenario', 'predicted_choice']], on=['idperson', 'scenario'], how='left').fillna(0)
df_sorted = df_sorted.merge(optimal_choice[['idperson', 'scenario', 'optimal_choice']], on=['idperson', 'scenario'], how='left').fillna(0)

# Convert binary indicators to int (to avoid float formatting)
df_sorted['predicted_choice'] = df_sorted['predicted_choice'].astype(int)
df_sorted['optimal_choice'] = df_sorted['optimal_choice'].astype(int)

# Now, df has 'predicted_choice' and 'optimal_choice' columns as binary indicators
# Display the first few rows to verify the sorting
df_sorted[['idperson','probability','likelihood','predicted_choice','optimal_choice','choice_made','computed_utility', 'scenario', 'lhw_h0','lhw_h1','lhw_h2','lhw_h3','lhw','ils_udb_yds','yivwg']].head(10)


,idperson,probability,likelihood,predicted_choice,optimal_choice,choice_made,computed_utility,scenario,lhw_h0,lhw_h1,lhw_h2,lhw_h3,lhw,ils_udb_yds,yivwg
0,87010001,0.005722,5.163456,0,0,0,6.272085e+05,h0,0,7,42,54,42,825.40,13.85
1,87010001,0.535728,0.624128,1,0,1,3.729029e+06,h2,0,7,42,54,42,2013.19,13.85
2,87010001,0.019588,3.932827,0,0,0,6.272097e+05,h1,0,7,42,54,42,825.40,13.85
3,87010001,0.366549,1.003622,0,1,0,5.682724e+06,h3,0,7,42,54,42,2485.32,13.85
4,87060001,0.480483,0.732963,1,0,1,5.393733e+06,h2,0,11,44,50,44,2421.29,17.05
5,87060001,0.005417,5.218274,0,0,0,6.502002e+05,h0,0,11,44,50,44,840.40,17.05
6,87060001,0.034210,3.375244,0,0,0,6.502020e+05,h1,0,11,44,50,44,840.40,17.05
7,87060001,0.390829,0.939485,0,1,0,6.640834e+06,h3,0,11,44,50,44,2686.71,17.05
8,87070001,0.012925,4.348601,0,0,1,2.066611e+06,h0,0,7,27,52,0,1498.60,6.68
9,87070001,0.446935,0.805342,1,1,0,5.968990e+06,h3,0,7,27,52,0,2547.16,6.68


In [63]:
import numpy as np

def simulate_unobserved_effects(n, sigma_zeta):
    """Simulate unobserved effects from a normal distribution."""
    return np.random.normal(0, sigma_zeta, n)

def utility(c, l, params, row, zeta):
    """Updated utility function including unobserved heterogeneity effect."""
    alpha1, alpha2, beta1, beta2, gamma = params[:5]
    observed_effect = sum(row[var] * params[i+5] for i, var in enumerate(observed_heterogeneity_vars))
    return (alpha1 + zeta) * c + alpha2 * c**2 + beta1 * l + beta2 * l**2 + gamma * c * l + observed_effect

def ind_likelihood(params, row, scenarios=['h0', 'h1', 'h2', 'h3'], n_simulations=100):
    """Updated individual likelihood function with simulated unobserved heterogeneity."""
    sigma_zeta = params[-1]  # Standard deviation of unobserved heterogeneity for alpha1
    simulated_likelihoods = []
    for _ in range(n_simulations):
        zeta = simulate_unobserved_effects(1, sigma_zeta)
        utilities = [utility(row['ils_udb_yds'], 80 - row[f'lhw_{scenario}'], params, row, zeta) for scenario in scenarios]
        chosen_index = scenarios.index(row['scenario'])
        log_sum_exp_utilities = logsumexp(utilities)
        simulated_likelihoods.append(utilities[chosen_index] - log_sum_exp_utilities)
    # Average the simulated likelihoods to approximate the integral over the unobserved heterogeneity distribution
    return -np.mean(simulated_likelihoods)


In [64]:
initial_params = [.766e+00,  9.197e-01,  1.135e-01, -1.993e-03,  1.893e-05] + [0.01] * len(observed_heterogeneity_vars) + [0.5]


In [65]:
from scipy.optimize import minimize

result = minimize(
    fun=lambda params: total_likelihood(params, df),
    x0=initial_params,
    method='L-BFGS-B'  # You might need to adjust the optimization method based on your specific needs
)


In [66]:
result

  message: CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
  success: True
   status: 0
      fun: 20242.855137401923
        x: [ 7.660e-01  9.197e-01  1.135e-01 -4.723e-04 -4.754e-05
             1.000e-02  1.000e-02  5.000e-01]
      nit: 15
      jac: [ 1.819e-03  5.093e-03 -3.665e+03  1.934e+03  6.172e+04
             7.640e-03 -1.455e-03  0.000e+00]
     nfev: 207
     njev: 23
 hess_inv: <8x8 LbfgsInvHessProduct with dtype=float64>

In [75]:
params = [
    7.660e-01,  # alpha1: Effect of consumption (c) on utility
    9.197e-01,  # alpha2: Effect of consumption squared (c^2) on utility
    1.135e-01,  # beta1: Effect of leisure (l) on utility
    -4.723e-04, # beta2: Effect of leisure squared (l^2) on utility
    -4.754e-05, # gamma: Interaction effect between consumption and leisure
    1.000e-02,  # Coefficient for the first observed heterogeneity variable
    1.000e-02,  # Coefficient for the second observed heterogeneity variable
    5.000e-01   # sigma_zeta: Standard deviation of unobserved heterogeneity for alpha1
]


In [76]:
import numpy as np
import pandas as pd
from scipy.special import logsumexp

# Assuming df is your DataFrame
# Assuming params contains your model parameters
# Assuming observed_heterogeneity_vars contains names of observed heterogeneity variables

# Step 1: Compute computed_utility
def compute_utility(row, params):
    c = row['ils_udb_yds']
    l = 80 - row['lhw']
    zeta = 0  # Assuming expected value of unobserved effects is zero for utility computation
    observed_effect = sum(row[var] * params[i+5] for i, var in enumerate(observed_heterogeneity_vars))
    alpha1, alpha2, beta1, beta2, gamma = params[:5]
    utility_value = (alpha1 + zeta) * c + alpha2 * c**2 + beta1 * l + beta2 * l**2 + gamma * c * l + observed_effect
    return utility_value

df['computed_utility'] = df.apply(compute_utility, params=params, axis=1)

# Step 2: Compute like column
def compute_like(row, df):
    utilities = df[df['idperson'] == row['idperson']]['computed_utility'].to_numpy()
    exp_utilities = np.exp(utilities)
    sum_exp_utilities = np.sum(exp_utilities)
    choice_probability = np.exp(row['computed_utility']) / sum_exp_utilities
    return choice_probability

df['like'] = df.apply(compute_like, df=df, axis=1)


C:\Users\hisham\AppData\Local\Temp\ipykernel_11652\2879441699.py:24: RuntimeWarning: overflow encountered in exp
  exp_utilities = np.exp(utilities)
C:\Users\hisham\AppData\Local\Temp\ipykernel_11652\2879441699.py:26: RuntimeWarning: overflow encountered in exp
  choice_probability = np.exp(row['computed_utility']) / sum_exp_utilities
C:\Users\hisham\AppData\Local\Temp\ipykernel_11652\2879441699.py:26: RuntimeWarning: invalid value encountered in scalar divide
  choice_probability = np.exp(row['computed_utility']) / sum_exp_utilities


In [77]:
df[['idperson', 'like' , 'computed_utility', 'choice_made']].head()

,idperson,like,computed_utility,choice_made
0,87010001,NaN,6.272130e+05,0
1,87060001,NaN,6.502046e+05,0
2,87070001,NaN,2.066613e+06,1
3,87090001,NaN,6.183883e+05,0
4,87170002,NaN,5.751745e+05,0


In [73]:

#values for exploration
exploration_params = [.766e+00,  9.197e-01,  1.135e-01, -1.993e-03,  4.341e-05]

# Compute utilities for each row in the DataFrame using the exploration parameter values
df['computed_utility'] = df.apply(calculate_utilities, axis=1, args=(exploration_params,))

# Print the first few rows to see the computed utilities
print(df[['idperson', 'scenario', 'computed_utility']].head())

TypeError: utility() missing 1 required positional argument: 'zeta'

In [72]:
# Assuming 'df' is your DataFrame with an 'idperson' and 'computed_utility' columns

# Step 1: Calculate the exponential of computed_utility for each row
df['exp_utility'] = np.exp(df['computed_utility'])

# Step 2: Group by 'idperson' and calculate the sum of exp_utility for each group
sum_exp_utility_by_person = df.groupby('idperson')['exp_utility'].transform('sum')

# Step 3: Calculate 'like' as the ratio of exp_utility to the sum of exp_utility for the corresponding idperson
df['like'] = df['exp_utility'] / sum_exp_utility_by_person

# You can drop the 'exp_utility' column if it's no longer needed
df = df.drop('exp_utility', axis=1)

# Verify the results
df[['computed_utility','like']].head()


c:\ProgramData\anaconda3\Lib\site-packages\pandas\core\arraylike.py:396: RuntimeWarning: overflow encountered in exp
  result = getattr(ufunc, method)(*inputs, **kwargs)


,computed_utility,like
0,6.272085e+05,NaN
1,6.502002e+05,NaN
2,2.066611e+06,NaN
3,6.183836e+05,NaN
4,5.751697e+05,NaN


In [ ]:
def softmax(utilities):
    e_u = np.exp(utilities - np.max(utilities))
    return e_u / e_u.sum()


In [ ]:
# Group by 'idperson' and calculate probabilities for each group
def calculate_probabilities(group):
    utilities = group['computed_utility'].values
    probabilities = softmax(utilities)
    group['probability'] = probabilities
    return group

df = df.groupby('idperson').apply(calculate_probabilities)


In [ ]:
# Reset the index to ensure 'idperson' is treated as a column.
df = df.reset_index(drop=True)


In [ ]:
# Ensure 'idperson' is one of the DataFrame's columns
assert 'idperson' in df.columns, "'idperson' must be a column in the DataFrame"


In [ ]:
# Define or import your 'calculate_probabilities' function here, ensuring it works as expected

# Apply groupby operation
df = df.groupby('idperson').apply(calculate_probabilities).reset_index(drop=True)

# Assuming 'calculate_probabilities' correctly calculates and assigns a 'probability' column
# Proceed to mark the optimal choice based on utilities and probabilities
df['choice_optimal'] = 0
df.loc[df.groupby('idperson')['computed_utility'].idxmax(), 'choice_optimal'] = 1

df['highest_probability'] = 0
df.loc[df.groupby('idperson')['probability'].idxmax(), 'highest_probability'] = 1


In [ ]:
df[['idperson','highest_probability','choice_optimal','ils_udb_yds','choice_made','computed_utility', 'scenario', 'lhw_h0','lhw_h1','lhw_h2','lhw_h3','lhw','yem','yivwg']].head(10)


In [ ]:
# This assumes 'choice_optimal' correctly identifies the optimal choice for each individual
# Let's add a 'predicted_choice' column to hold this information
choices = ['h0', 'h1', 'h2', 'h3']
df['predicted_choice'] = None
for choice in choices:
    df.loc[df['choice_optimal'] == 1, 'predicted_choice'] = choice


In [ ]:
# Assuming df is your DataFrame and it contains 'scenario' and 'choice_made' columns
df_choices_made = df[df['choice_made'] == 1]


In [ ]:
import matplotlib.pyplot as plt

# Count the occurrences of each scenario where a choice was made
choice_counts = df_choices_made['scenario'].value_counts().sort_index()

# Create a bar plot/histogram
plt.figure(figsize=(10, 6))
choice_counts.plot(kind='bar', color='skyblue')
plt.title('Distribution of Choices Made Across Scenarios')
plt.xlabel('Scenario')
plt.ylabel('Number of Choices Made')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--')

# Show the plot
plt.show()


In [ ]:
# Assuming df is your DataFrame and it contains 'scenario' and 'choice_made' columns
df_choices_per = df[df['choice_optimal'] == 1]

In [ ]:
import matplotlib.pyplot as plt

# Count the occurrences of each scenario where a choice was made
choice_counts = df_choices_per['scenario'].value_counts().sort_index()

# Create a bar plot/histogram
plt.figure(figsize=(10, 6))
choice_counts.plot(kind='bar', color='skyblue')
plt.title('Distribution of Choices pred Across Scenarios')
plt.xlabel('Scenario')
plt.ylabel('Number of Choices Made')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--')

# Show the plot
plt.show()


In [ ]:
choice_made_counts = df.groupby('scenario')['choice_made'].sum().reset_index()
choice_made_counts.rename(columns={'choice_made': 'count_choice_made'}, inplace=True)

# If you need to first create a 'predicted_choice' column based on 'choice_optimal', follow this pattern:
# df['predicted_choice'] = [some logic to determine predicted choice]

# Then, count how many times each choice was predicted as optimal
choice_optimal_counts = df[df['choice_optimal'] == 1]['predicted_choice'].value_counts().reset_index()
choice_optimal_counts.columns = ['scenario', 'count_choice_optimal']


In [ ]:
# Merge the counts together based on the scenario
choice_counts = pd.merge(choice_made_counts, choice_optimal_counts, on='scenario', how='outer').fillna(0)


In [ ]:
import matplotlib.pyplot as plt

choice_counts.plot(x='scenario', kind='bar', figsize=(10, 6))
plt.title('Comparison of Choice Made and Optimal Choice Counts')
plt.xlabel('Scenario')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.legend(['Choices Made', 'Optimal Choices'])
plt.grid(axis='y', linestyle='--')
plt.show()


In [ ]:
df[['idperson','highest_probability','choice_optimal','ils_udb_yds','choice_made','computed_utility', 'scenario', 'lhw_h0','lhw_h1','lhw_h2','lhw_h3','lhw','yem','yivwg']].to_csv('estimated.csv')

In [ ]:
df_chosen = df[df['choice_made'] == 1]
# Count the number of correct predictions
correct_predictions = df_chosen[df_chosen['choice_optimal'] == 1].shape[0]
print(f"Number of Correctly Predicted Choices: {correct_predictions}")


In [ ]:
# Assuming 'predicted_choice' indicates the model's optimal choice prediction
# And 'scenario' is the actual choice made
correct_predictions = df_chosen[df_chosen['scenario'] == df_chosen['predicted_choice']].shape[0]
print(f"Number of Correctly Predicted Choices: {correct_predictions}")


In [ ]:
df_pred1 = df[df['predicted_choice'] == 'h3' ]

In [ ]:
df_pred1['predicted_choice'].describe

In [ ]:
bingo = df_chosen[df_chosen['choice_optimal'] == 1]
bingo[['choice_made', 'choice_optimal', 'lhw', 'computed_utility','predicted_choice','scenario']]

In [ ]:
xxx = bingo[bingo ['scenario'] != 'h3']

xxx[['choice_made', 'choice_optimal', 'lhw', 'computed_utility','predicted_choice','scenario']]

In [ ]:
def utility(c, l, alpha1, alpha2, beta1, beta2, gamma, heterogeneity_vars):
    """
    Quadratic utility function with interaction between c and l, and additional individual heterogeneity.
    
    Parameters:
    - c: Consumption
    - l: Leisure
    - alpha1, alpha2: Parameters for consumption
    - beta1, beta2: Parameters for leisure
    - gamma: Interaction parameter between consumption and leisure
    - heterogeneity_vars: Additional individual-specific variables influencing utility
    """
    # Incorporate heterogeneity variables into the utility calculation as needed
    # This could involve adding terms to the utility function or modifying existing ones
    # Example: return alpha1 * c + alpha2 * c**2 + beta1 * l + beta2 * l**2 + gamma * c * l + heterogeneity_effect
    heterogeneity_effect = sum([coef * var for coef, var in heterogeneity_vars])
    return alpha1 * c + alpha2 * c**2 + beta1 * l + beta2 * l**2 + gamma * c * l + heterogeneity_effect


In [ ]:
def ind_likelihood(params, yds, lhw_actual, lhw_scenario, choice_made,heterogeneity_vars):
    alpha1, alpha2, beta1, beta2, gamma = params  # Updated to match the new utility function
    utilities = [
        utility(yds, 80 - lhw_actual, alpha1, alpha2, beta1, beta2, gamma, heterogeneity_vars),
        utility(yds, 80 - lhw_scenario, alpha1, alpha2, beta1, beta2, gamma,heterogeneity_vars)
    ]
    # Continue with the log-sum-exp and return the negative log likelihood
    log_probabilities = utilities - logsumexp(utilities)
    return -log_probabilities[int(choice_made)]

In [ ]:
def total_likelihood(params, df):
    total_ll = 0
    for _, row in df.iterrows():
        # Build heterogeneity_vars from the row; example assumes heterogeneity_vars is a list of (coef, var) tuples
        heterogeneity_vars = [(param, row[var_name]) for param, var_name in zip(params[5:], heterogeneity_var_names)]
        total_ll += ind_likelihood(params, row, heterogeneity_vars)
    return total_ll

